# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [ ]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [ ]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

In [ ]:
links = fetch_website_links("https://edwarddonner.com")
links

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://edwarddonner.com"))

In [ ]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
select_relevant_links("https://huggingface.co")

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

In [ ]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure("HuggingFace", "https://huggingface.co")

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from playwright.async_api import async_playwright, Playwright
from utils import search_book, go_to_author_page, fetch_website_links, fetch_website_contents



In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key) > 10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")



API key looks good so far


In [3]:
async def run_playwright(playwright: Playwright, author_name: str, url: str):
    browser = await playwright.chromium.launch(headless=False)
    page = await browser.new_page()
    #Launch browser , open new tab and go to goodreads.com
    await page.goto(url)

    # Search for the book in goodreads apnd get url of details page
    book_url = await search_book(page, author_name)
    author_page = await go_to_author_page(page, author_name)
    return author_page



async def run(bookname: str, url:str) -> str:
    async with async_playwright() as playwright:
        author_page = await run_playwright(playwright, bookname, url)
        return author_page

# author_name = input("Type the name of the author you want to create a portfolio for:")
# await run(author_name, "https://www.goodreads.com/")

In [4]:
MODEL = 'gpt-5-nano'
openai = OpenAI()

author_name = input("Type the name of the author you want a portfolio for:")
author_page = await run(author_name, "https://www.goodreads.com/")
# links = fetch_website_links(author_page)
## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.
link_system_prompt = """
You are provided with a list of links found on a goodreads author page.
You are able to decide which of the links would be most relevant to include in a portfolio about the author,
such as author's webpage, link to their literary works, contat details, award links etc.,
You should respond in JSON as in this example:

{
    "links": [
        {"type": "author page", "url": "https://full.url/goes/here/author_website"},
        {"type": "author's books", "url": ["/book/show/book1", "/book/show/book2"] }
    ]
}
"""

def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the goodreads author page {url} -
Please decide which of these are relevant web links for a portfolio about the author,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, videos links and  navigation links

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt


print(get_links_user_prompt(author_page))



Sign-in popup detected! Closing it...

Here is the list of links on the goodreads author page https://www.goodreads.com/author/show/3541.Barbara_Kingsolver -
Please decide which of these are relevant web links for a portfolio about the author,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, videos links and  navigation links

Links (some might be relative links):

https://www.goodreads.com/blog/show/3088?ref=bbsummer_eb
/
/?ref=nav_home
/review/list?ref=nav_mybooks
/book?ref=nav_brws
/recommendations?ref=nav_brws_recs
/choiceawards?ref=nav_brws_gca
/genres?ref=nav_brws_genres
/giveaway?ref=nav_brws_giveaways
/book/popular_by_date/2026/5?ref=nav_brws_newrels
/list?ref=nav_brws_lists
/book?ref=nav_brws_explore
/news?ref=nav_brws_news
/genres/art
/genres/biography
/genres/business
/genres/children-s
/genres/christian
/genres/classics
/genres/comics
/genres/cookbooks
/genres/ebooks
/genres/fantasy
/genres/fiction
/genres/graphic-novels
/genres/hist

In [5]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links


print(select_relevant_links(author_page))

Selecting relevant links for https://www.goodreads.com/author/show/3541.Barbara_Kingsolver by calling gpt-5-nano
Found 4 relevant links
{'links': [{'type': 'author website', 'url': 'https://barbarakingsolver.net/'}, {'type': "author's books", 'url': ['https://www.goodreads.com/book/show/7244.The_Poisonwood_Bible', 'https://www.goodreads.com/book/show/30868.The_Bean_Trees', 'https://www.goodreads.com/book/show/14249.Prodigal_Summer', 'https://www.goodreads.com/book/show/13438524-flight-behavior', 'https://www.goodreads.com/book/show/77262.Animal_Dreams', 'https://www.goodreads.com/book/show/60194162-demon-copperhead', 'https://www.goodreads.com/book/show/37959904-unsheltered', 'https://www.goodreads.com/book/show/6433752-the-lacuna', 'https://www.goodreads.com/book/show/14250.Pigs_in_Heaven']}, {'type': 'press / media', 'url': ['https://www.southernliving.com/barbara-kingsolver-on-heart-of-appalachia-11719582']}, {'type': 'article about author on author site', 'url': ['https://barbaraki

In [6]:

def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        if isinstance(link["url"], list):
            for book_link in link["url"]:
                result += fetch_website_contents(book_link)
        else:
            result += fetch_website_contents(link["url"])
    return result


print(fetch_page_and_all_relevant_links(author_page))

Selecting relevant links for https://www.goodreads.com/author/show/3541.Barbara_Kingsolver by calling gpt-5-nano
Found 5 relevant links
## Landing Page:

Barbara Kingsolver (Author of Demon Copperhead)

Home
My Books
Browse ▾
Recommendations
Choice Awards
Genres
Giveaways
New Releases
Lists
Explore
News & Interviews
Genres
Art
Biography
Business
Children's
Christian
Classics
Comics
Cookbooks
Ebooks
Fantasy
Fiction
Graphic Novels
Historical Fiction
History
Horror
Memoir
Music
Mystery
Nonfiction
Poetry
Psychology
Romance
Science
Science Fiction
Self Help
Sports
Thriller
Travel
Young Adult
More Genres
Community ▾
Groups
Quotes
Ask the Author
Sign In
Join
Sign up
View profile
Profile
Friends
Groups
Discussions
Comments
Reading Challenge
Kindle Notes & Highlights
Quotes
Favorite genres
Friends’ recommendations
Account settings
Help
Sign out
Home
My Books
Browse ▾
Recommendations
Choice Awards
Genres
Giveaways
New Releases
Lists
Explore
News & Interviews
Genres
Art
Biography
Business
Childre

In [7]:

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages of an author from goodreads
and creates a short portfolio about the author for prospective publishers, clients or potential employers.
Respond in markdown without code blocks.
Include bio of author, contact details, published works, awards and accolades if you have the information.
"""


# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

def get_brochure_user_prompt(author_name, url):
    user_prompt = f"""
You are looking at an author named: {author_name}
Here are the contents of their landing page and other relevant pages;
use this information to build a short portfolio of the author in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000]  # Truncate if more than 5,000 characters
    return user_prompt


get_brochure_user_prompt(author_name, author_page)

Selecting relevant links for https://www.goodreads.com/author/show/3541.Barbara_Kingsolver by calling gpt-5-nano
Found 5 relevant links


"\nYou are looking at an author named: barbara kingsolver\n\nHere are the contents of their landing page and other relevant pages;\nuse this information to build a short portfolio of the author in markdown without code blocks.\n\n\n## Landing Page:\n\nBarbara Kingsolver (Author of Demon Copperhead)\n\nHome\nMy Books\nBrowse ▾\nRecommendations\nChoice Awards\nGenres\nGiveaways\nNew Releases\nLists\nExplore\nNews & Interviews\nGenres\nArt\nBiography\nBusiness\nChildren's\nChristian\nClassics\nComics\nCookbooks\nEbooks\nFantasy\nFiction\nGraphic Novels\nHistorical Fiction\nHistory\nHorror\nMemoir\nMusic\nMystery\nNonfiction\nPoetry\nPsychology\nRomance\nScience\nScience Fiction\nSelf Help\nSports\nThriller\nTravel\nYoung Adult\nMore Genres\nCommunity ▾\nGroups\nQuotes\nAsk the Author\nSign In\nJoin\nSign up\nView profile\nProfile\nFriends\nGroups\nDiscussions\nComments\nReading Challenge\nKindle Notes & Highlights\nQuotes\nFavorite genres\nFriends’ recommendations\nAccount settings\nHelp\

In [8]:
def create_brochure(author_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(author_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))


create_brochure(author_name, author_page)

Selecting relevant links for https://www.goodreads.com/author/show/3541.Barbara_Kingsolver by calling gpt-5-nano
Found 14 relevant links


# Barbara Kingsolver – Author Portfolio

## Biography
Barbara Ellen Kingsolver, born April 8, 1955, in Annapolis, Maryland, United States, is an acclaimed American novelist, essayist, and poet. She is renowned for weaving themes of social justice, biodiversity, and environmental awareness into her work. Kingsolver's literary style spans fiction, nonfiction, poetry, critical commentary, and journalism. She pursues bold, innovative storytelling, embracing the challenge of trying something new with each book.

## Career Highlights & Awards
- **Pulitzer Prize for Fiction (2023)** for her novel *Demon Copperhead*.
- Widely recognized for *The Poisonwood Bible*, a gripping tale of a missionary family in the Congo.
- Author of *Animal, Vegetable, Miracle*, a nonfiction narrative chronicling her family's experiment with eating locally.
- Kingsolver’s work often explores complex social and environmental issues, making a significant impact in contemporary literature.

## Selected Published Works
- *Demon Copperhead* (2023)
- *The Poisonwood Bible* 
- *Animal, Vegetable, Miracle* 
- *How to Fly (In Ten Thousand Easy Lessons)* (Poetry collection)

## Writing Style & Themes
- Focus on **social justice**, **biodiversity**, and **environmental sustainability**.
- Combines **fiction**, **nonfiction**, and **poetry**.
- Known for richly developed characters and evocative prose.
- Engages readers with insightful commentary on humanity and ecology.

## Online Presence & Contact
- Official Website: [barbarakingsolver.net](http://barbarakingsolver.net/)
- The website offers extensive information including biography, bibliography, awards, media kit, FAQ, news & events, and contact details.

## Notable Quotes by Barbara Kingsolver
- “What keeps me awake at the wheel is the thrill of trying something completely new with each book.”
- “The trees exhaled in communion, rode their new continents, survived the end of the world.” — from *Forests of Antarctica*
- “Remember to leave a window open, oven door closed, stones on the ground not in your pockets. Maybe just one precious in a fist, or against a hot cheek.” — from *Dancing with the Devil: Advice for the Female Poet*

Barbara Kingsolver remains a powerful voice in contemporary literature, celebrated for her commitment to meaningful storytelling that touches on urgent global issues through compelling narratives and poetic expression.

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


stream_brochure("HuggingFace", "https://huggingface.co")
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")
